In [ ]:
import sys, os
from pathlib import Path
_root = Path(os.getcwd())
while not ((_root / 'src').exists() and (_root / 'pyproject.toml').exists()) and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

# Atenção residual às manchetes — experimento rápido

**Demonstração exploratória, não prova de alfa.** Execute com GPU T4 ou L4 no Google Colab. Antes, conclua `fast_fusion_demo_colab.ipynb`: esta execução lê `fast_demo_panel.npz`, `fast_demo_events.npz`, `ts_only.pt` e `fast_demo_provenance.json` de `MyDrive/latent_fusion_data/fast_fusion_demo_results/`. Os embeddings MOMENT-1-small e MiniLM foram produzidos naquela execução; **não serão gerados ou baixados novamente**. O notebook não necessita de `momentfm`, `sentence-transformers` nem LLM.

A partir do **mesmo modelo numérico congelado**, compara: série temporal, série + atenção apenas à idade/contagem das manchetes e série + atenção ao conteúdo textual. Os resíduos começam em zero, são limitados e não alteram a previsão em dias sem notícia recente. Treino e seleção de épocas usam somente treino/validação cronológicos; o teste de 2022–23 já foi inspecionado e continua exploratório.

In [ ]:
import subprocess
from google.colab import drive
import torch
drive.mount('/content/drive')
if not torch.cuda.is_available():
    raise RuntimeError('Ative uma GPU T4 ou L4: Ambiente de execução → Alterar tipo de ambiente')
REPO = Path('/content/latent_fusion')
if REPO.exists():
    if not (REPO / '.git').is_dir():
        raise RuntimeError(f'Pasta existente não é repositório Git: {REPO}')
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/rbfcdog/latent_fusion.git', str(REPO)], check=True)
SOURCE = Path('/content/drive/MyDrive/latent_fusion_data/fast_fusion_demo_results')
OUT = Path('/content/drive/MyDrive/latent_fusion_data/fast_residual_attention_results')
for name in ('fast_demo_panel.npz', 'fast_demo_events.npz', 'ts_only.pt', 'fast_demo_provenance.json'):
    if not (SOURCE / name).is_file():
        raise FileNotFoundError(f'Execute primeiro fast_fusion_demo_colab.ipynb: {SOURCE / name}')
sys.path.insert(0, str(REPO))
{'gpu': torch.cuda.get_device_name(0), 'embeddings': str(SOURCE), 'saída': str(OUT)}

## Treino somente do residual

O módulo confirma que embeddings e checkpoint são da mesma execução. A atenção usa a representação numérica congelada como consulta; a ablação de metadados preserva exatamente as datas, máscaras e número de manchetes, mas fornece somente a idade de cada uma. Os alvos são retornos de cinco pregões em excesso da média dos oito ativos. A parada antecipada considera MSE na validação **com notícia**, e o teste avalia todos os pregões. As carteiras são long-only, com rebalanceamento a cada cinco pregões, custo de 10 bps e comparação contra comprar e manter os mesmos oito ativos.

In [ ]:
from src.fusion.fast_residual_experiment import run_experiment
result = run_experiment(SOURCE, OUT, cost_bps=10)
{'arquivos': str(result['output']), 'históricos': [str(OUT / f'{model}_history.csv') for model in result['histories']]}

## Comparação no mesmo teste

Compare MSE e IC tanto com o numérico quanto com a previsão de excesso zero; compare retorno líquido, alfa, beta e curva de capital com comprar e manter. A vantagem sobre a ablação idade/contagem é necessária para atribuir algum valor ao **conteúdo** das manchetes. Mesmo uma melhora aqui não confirma desempenho futuro: universo selecionado ex post, notícia indisponível após novembro de 2023 e teste já examinado.

In [ ]:
from IPython.display import Image, display
display(result['signal'], result['portfolio'])
display(result['histories']['metadata'], result['histories']['text_residual'])
display(Image(filename=str(OUT / 'residual_vs_bh.png')))
{'relatório': str(OUT / 'results.md'), 'curvas': str(OUT / 'equity_curves.csv'), 'proveniência': str(OUT / 'run_config.json')}